# Model Design
----

## Review
- Datasets, Dataloaders and Transforms
- We mightt want to perform additional transformation for a more sensible features
  - feature transformation to get `home_size=total_rooms/households`, `home_density=total_bedrooms/total_rooms`,`household_size=population/households`

In [ ]:
import torch
from torch import nn
import pandas as pd

from torch.utils.data import Dataset, DataLoader

from pathlib import Path

### Feature Engineering
- Feature Transformation
- Normalization
- Feature Selection

In [ ]:
from sklearn.preprocessing import StandardScaler
import os
import numpy as np

In [ ]:
  def feature_transform(df):
    # this internal method will serve as transform method for both the features and atarges
    df['home_size'] = df['total_rooms']/df['households']
    df['home_density'] = df['total_bedrooms']/df['total_rooms']
    df['household_size'] = df['population']/df['households']
    return df[['longitude','latitude','housing_median_age','median_income','home_size','home_density','household_size','median_house_value']]

In [ ]:
dataset_path = Path("/content/sample_data")

train_df = pd.read_csv(dataset_path/"california_housing_train.csv")
test_df = pd.read_csv(dataset_path/"california_housing_test.csv")

# Feature Transform
train_df = feature_transform(train_df)
test_df = feature_transform(test_df)

print(test_df.columns)


# Normalization (we accept outliers for now)
## Fit scaler to train only to avoid data leakage
scaler = StandardScaler()
scaler.fit(train_df)

train_np = scaler.transform(train_df)
test_np = scaler.transform(test_df)

# Save as preprocessed data
PREPROCESSED_DIR = Path("/content/preprocessed_data")
os.makedirs(name=PREPROCESSED_DIR, exist_ok=True)
np.savez_compressed(PREPROCESSED_DIR/"train.npz", data=train_np, )
np.savez_compressed(PREPROCESSED_DIR/"test.npz", data=test_np)


Index(['longitude', 'latitude', 'housing_median_age', 'median_income',
       'home_size', 'home_density', 'household_size', 'median_house_value'],
      dtype='object')


In [ ]:
# test load
# with np.load(PREPROCESSED_DIR/"train.npz") as data:
#     X = data['data']
#     print(X[:, :8].shape)

In [ ]:
## Dataset

def to_tensor(x):
  return torch.from_numpy(x)

class HousingDataset(Dataset):
  def __init__(self, root: Path, train: bool = True, transform=None, target_transform=None):
    with np.load(PREPROCESSED_DIR/f"{'train' if train else 'test' }.npz") as data:
        X,y = data['data'][:,:7], data['data'][:,7:8]

    if transform:
      self.X = transform(X)
    if target_transform:
      self.y = transform(y)


  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx,], self.y[idx,]


train_dataset = HousingDataset(dataset_path, train=True, transform=to_tensor, target_transform=to_tensor)
test_dataset = HousingDataset(dataset_path, train=False, transform=to_tensor, target_transform=to_tensor)

### Hyperparameters
- batch_size=32
- epochs=8
- learning_rate=1e-2

In [ ]:
batch_size=32
epochs=8
learning_rate=1e-2

In [ ]:
# for idx, (X, y) in enumerate(test_dataset):
#   print(idx)
#   print(X.shape, X)
#   print(y.shape, y)
#   break

In [ ]:
train_dataloader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
for X,y in train_dataloader:
  print(X.shape, y.shape)
  break

torch.Size([32, 7]) torch.Size([32, 1])


## Building a Neural Network
Here we want to build a `Regression` model that identifies the `median house value` of a house based on final features.

In [ ]:
class Regressor(nn.Module):
  def __init__(self, in_features, out_features, hidden_features):
    # 3 layer dense network
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(in_features=in_features, out_features=hidden_features),
        nn.ReLU(),
        nn.Linear(hidden_features, hidden_features//2),
        nn.ReLU(),
        nn.Linear(hidden_features//2, hidden_features//2),
        nn.ReLU(),
        nn.Linear(hidden_features//2, out_features)
    )

  def forward(self, X):
    return self.layers(X)

model = Regressor(7, 1, 16)

In [ ]:
for params in model.parameters():
  print(params.shape)

torch.Size([16, 7])
torch.Size([16])
torch.Size([8, 16])
torch.Size([8])
torch.Size([8, 8])
torch.Size([8])
torch.Size([1, 8])
torch.Size([1])


In [ ]:
# random inference
X=torch.rand(5,7)
model(X)

tensor([[0.2787],
        [0.3050],
        [0.2902],
        [0.2826],
        [0.2911]], grad_fn=<AddmmBackward0>)

## Training
- we use the MSE loss function since this is a regression problem.

In [ ]:
# define the loss function
loss_fn = nn.MSELoss()
# define the optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
# the training loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(epochs):
  model.train()
  for batch, (X,y) in enumerate(train_dataloader):
    # print(f"Epoch {epoch} | Batch {batch}")

    # Cast input tensors to float32
    X,y = X.to(device).float(), y.to(device).float()
    # print(X.dtype, y.dtype)

    pred = model(X)
    loss = loss_fn(pred, y)

    optimizer.zero_grad() #reset grad
    loss.backward() # compute gradient,
    optimizer.step() #update params

    # print(f"Loss: {loss.item()}")

  # compute validation loss
  with torch.no_grad():
    model.eval()
    X_test = test_dataset.X.to(device).float()
    y_test = test_dataset.y.to(device).float()
    pred = model(X_test)
    loss = loss_fn(pred, y_test)
    print(f"Validation Loss for epoch {epoch}: {loss.item()}")
    pass


Validation Loss for epoch 0: 0.270588219165802
Validation Loss for epoch 1: 0.2711910605430603
Validation Loss for epoch 2: 0.2687842547893524
Validation Loss for epoch 3: 0.26772502064704895
Validation Loss for epoch 4: 0.2594236135482788
Validation Loss for epoch 5: 0.25904586911201477
Validation Loss for epoch 6: 0.2607308626174927
Validation Loss for epoch 7: 0.2606014609336853
